## Einleitung

Im Rahmen der Mini-Challenge des Moduls „Applied Machine Learning“ liegt der Fokus auf der Entwicklung eines Modells zur Unterstützung einer Werbekampagne für das Cross-Selling von Kreditkarten. Ziel ist es, mithilfe prädiktiver Modellierung die Affinität potenzieller Kunden zu Kreditkarten zu ermitteln. Es sollen dabei ausschließlich Kreditkarten des Typs „Classic“ oder „Gold“ betrachtet und Junior-Karten explizit ausgeschlossen werden.

Folgende zentrale Schritte sind Teil der analytischen Aufgabe:

1. **Datenaufbereitung**: Integration verschiedener Datenquellen, Bereinigung und Qualitätsprüfung der Daten.
2. **Feature Engineering**: Entwicklung relevanter Merkmale aus der Kundenhistorie.
3. **Modellentwicklung**: Aufbau und Validierung verschiedener Machine Learning Modelle.
4. **Evaluation und Selektion**: Bewertung und Vergleich der Modelle zur Auswahl des optimalen Modells.
5. **Interpretation und Einsatzplanung**: Übersetzung der Ergebnisse für praktische Anwendungen und Marketingmaßnahmen.

## Beschreibung der Datenbasis

Für die Modellierung stehen mehrere Datenquellen zur Verfügung, welche gemeinsam kombiniert werden müssen, um einen aussagekräftigen Modellierungsdatensatz zu erhalten:

- **Account**: Grundlegende Kontoinformationen wie Eröffnungsdatum und Ausstellungsfrequenz der Kontoauszüge.
- **Client**: Persönliche Kundendaten, insbesondere Geburtsdatum und Wohnbezirk.
- **Disposition**: Verknüpfung zwischen Kunden und Konten, inkl. Unterscheidung nach Inhaber oder Nutzer.
- **Transaction**: Transaktionen inklusive Beträge, Zeitpunkte und Charakterisierung der Transaktionen.
- **Permanent Order**: Daueraufträge, die von Kundenkonten ausgehen.
- **Loan**: Kredite, welche Kunden gewährt wurden, inklusive Rückzahlungsstatus.
- **Credit Card**: Informationen über ausgegebene Kreditkarten (nur Classic und Gold).
- **District**: Demografische Informationen, welche zusätzliche Merkmale zum Wohnbezirk der Kunden liefern.

### Tabellenverknüpfungen

Wenn die Tabellen gemäss Beschreibung verknüpft werden ergibt sich folgendes ERD ( entity-relationship diagram ):

![ERD before](img/erd_before.png)
( erstellt mit mermaidchart.com )

Eine genaue Untersuchung der Tabellen zeigt jedoch, dass sich in der Beschreibung Fehler eingeschlichen haben.  
Folgende Korrekturen sind nötig:

### Wichtige Korrekturen:

✅ Die Dispositionstabelle (DISP) stellt Viele-zu-Viele-Beziehung zwischen Kunden und Konten dar.  
✅ Kreditkarten (CARD) werden einer Disposition ausgestellt, nicht direkt einem Konto.  
✅ Kredite (LOAN) sind mit Konten verknüpft, nicht mit Kunden.  
✅ Daueraufträge (ORDER) werden vom Kontoinhaber erstellt.  
✅ Transaktionen (TRANS) sind mit Konten verknüpft, um finanzielle Aktivitäten zu verfolgen.  
✅ Bezirke (DISTRICT) beeinflussen sowohl Kunden (Wohnsitz) als auch Konten (Filialstandort).  

![ERD before](img/erd_after.png)
( erstellt mit mermaidchart.com )

## Daten Laden, Erste Exploration und Korrekturen

Zunächst soll die Datenqualität und Struktur der einzelnen Dateien (z. B. ACCOUNT, CLIENT, TRANS, LOAN, CARD, DISTRICT) geprüft werden. Es wird untersucht, ob alle relevanten Datensätze vorliegen, wie groß die Dateien sind und welche ersten Auffälligkeiten es gibt.

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Datenpfad
data_path = 'xselling_banking_data/'

# Laden der Dateien
try:
    df_account = pd.read_csv(data_path + 'ACCOUNT.CSV', sep=';', quotechar='"')
    df_client = pd.read_csv(data_path + 'CLIENT.CSV', sep=';', quotechar='"')
    df_disp   = pd.read_csv(data_path + 'DISP.CSV', sep=';', quotechar='"')
    df_order = pd.read_csv(data_path + 'ORDER.CSV', delimiter=';', header=None, skiprows=1, names=['order_id', 'account_id', 'bank_to', 'account_to', 'amount', 'k_symbol'])
    df_trans = pd.read_csv(data_path + 'TRANS.CSV', delimiter=';', header=None, skiprows=1, names=['trans_id', 'account_id', 'date', 'type', 'operation', 'amount', 'balance', 'k_symbol', 'bank', 'account'],
                           dtype={'bank': str}, quotechar='"', low_memory=False)
    df_loan = pd.read_csv(data_path + 'LOAN.CSV', delimiter=';', header=None, skiprows=1, names=['loan_id', 'account_id', 'date', 'amount', 'duration', 'payments', 'status'])
    df_card = pd.read_csv(data_path + 'CARD.CSV', delimiter=';', header=None, skiprows=1, names=['card_id', 'disp_id', 'type', 'issued'])
    df_district = pd.read_csv(data_path + 'DISTRICT.CSV', delimiter=';', header=None,  skiprows=1,
                              names=['district_id', 'district_name', 'region', 'no_inhabitants', 'municipalities_<500', 'municipalities_500_1999',
                                     'municipalities_2000_9999', 'municipalities_>10000', 'no_cities', 'urban_ratio', 'avg_salary',
                                     'unemployment_95', 'unemployment_96', 'entrepreneurs_per_1000', 'crimes_95', 'crimes_96'])
except Exception as e:
    print('Fehler beim Laden der Daten:', e)

# Erste Exploration: Größe der DataFrames
print('Anzahl Accounts:', df_account.shape)
print('Anzahl Clients:', df_client.shape)
print('Anzahl Dispositionen:', df_disp.shape)
print('Anzahl Orders:', df_order.shape)
print('Anzahl Transaktionen:', df_trans.shape)
print('Anzahl Loans:', df_loan.shape)
print('Anzahl Credit Cards:', df_card.shape)
print('Anzahl Distrikte:', df_district.shape)

Anzahl Accounts: (4500, 4)
Anzahl Clients: (5369, 3)
Anzahl Dispositionen: (5369, 4)
Anzahl Orders: (6471, 6)
Anzahl Transaktionen: (1056320, 10)
Anzahl Loans: (682, 7)
Anzahl Credit Cards: (892, 4)
Anzahl Distrikte: (77, 16)


Die Daten wurden korrekt eingelesen und die Anzahl Zeilen stimmt ( nach einigen Versuchen und Manipulationen ) mit der Beschreibung überein.

In [28]:
# Basic EDA für alle DataFrames
def basic_eda(df, df_name="DataFrame", show_plots=False):
    """
    Führt eine einfache EDA (Exploratory Data Analysis) auf einem Pandas DataFrame durch
    und gibt relevante Informationen aus.

    Parameter:
    -----------
    df : pd.DataFrame
        Das DataFrame, das analysiert werden soll
    df_name : str
        Name des DataFrames, wird bei der Ausgabe verwendet
    show_plots : bool
        Wenn True, werden zusätzlich einfache Histogramme 
        für numerische Features sowie Countplots für kategorische 
        Variablen erzeugt.
    """
    print(f"\n=== [ {df_name} ] ===")
    
    # 1) Shape
    print(f"Shape (Rows x Columns): {df.shape}")

    # 2) Datentypen und Speicherplatz
    print("\nInfo:")
    print(df.info())
    
    # 3) Erste Zeilen
    print("\nErste Zeilen:")
    print(df.head())

    # 4) Statistische Kurzdarstellung
    print("\nStatistische Übersicht (describe):")
    print(df.describe(include="all"))
    
    # 5) Fehlende Werte
    missing_vals = df.isnull().sum()
    print("\nAnzahl fehlender Werte pro Spalte:")
    print(missing_vals)
    
    # 6) Grundlegende Verteilungen (optional)
    if show_plots:
        numeric_cols = df.select_dtypes(include=["int", "float", "int64", "float64"]).columns
        cat_cols = df.select_dtypes(include=["object", "category"]).columns
        
        # Histogramme für numerische Spalten
        if len(numeric_cols) > 0:
            df[numeric_cols].hist(figsize=(10, 6), bins=30, layout=(1, len(numeric_cols)), edgecolor='black')
            plt.suptitle(f"{df_name} - Numerische Verteilungen", y=1.02, fontsize=14)
            plt.tight_layout()
            plt.show()
        
        # Countplots für kategorische Spalten (falls wenige Kategorien)
        for col in cat_cols:
            # Damit der Plot nicht zu überladen wird, 
            # z.B. nur anzeigen, wenn <= 10 unique Categories.
            if df[col].nunique() <= 10:
                sns.countplot(x=col, data=df)
                plt.title(f"{df_name} - Countplot für '{col}'")
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()

### Accounts

In [29]:
basic_eda(df_account, df_name="Accounts", show_plots=False)


=== [ Accounts ] ===
Shape (Rows x Columns): (4500, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   account_id   4500 non-null   int64 
 1   district_id  4500 non-null   int64 
 2   frequency    4500 non-null   object
 3   date         4500 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 140.8+ KB
None

Erste Zeilen:
   account_id  district_id         frequency    date
0         576           55  POPLATEK MESICNE  930101
1        3818           74  POPLATEK MESICNE  930101
2         704           55  POPLATEK MESICNE  930101
3        2378           16  POPLATEK MESICNE  930101
4        2632           24  POPLATEK MESICNE  930102

Statistische Übersicht (describe):
          account_id  district_id         frequency           date
count    4500.000000  4500.000000              4500    4500.000000
unique           NaN     

In [30]:
# 1) Spalte date in datetime umwandeln
df_account["date"] = pd.to_datetime(df_account["date"], format="%y%m%d")

# 2) Spalte district_id in int umwandeln
df_account["district_id"] = df_account["district_id"].astype(int)

# 3) Spalte frequency umbenennen
freq_map = {
        "POPLATEK MESICNE": "monthly",
        "POPLATEK TYDNE": "weekly",
        "POPLATEK PO OBRATU": "after_transaction"
    }
df_account["frequency"] = df_account["frequency"].map(freq_map)

# 4) Spalte frequency in Kategorien umwandeln
df_account["frequency"] = df_account["frequency"].astype("category")

# 5) Spalte date umbenennen
df_account.rename(columns={"date": "date_created"}, inplace=True)

# 6) Prüfen, ob account_id eindeutig ist
print("Anzahl eindeutiger account_id:", df_account["account_id"].nunique())
print("Anzahl Zeilen:", df_account.shape[0])

# 7) Zeitraum der Daten prüfen
print("Minimales Datum:", df_account["date_created"].min())
print("Maximales Datum:", df_account["date_created"].max())

Anzahl eindeutiger account_id: 4500
Anzahl Zeilen: 4500
Minimales Datum: 1993-01-01 00:00:00
Maximales Datum: 1997-12-29 00:00:00


In [31]:
basic_eda(df_account, df_name="Accounts", show_plots=False)


=== [ Accounts ] ===
Shape (Rows x Columns): (4500, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   account_id    4500 non-null   int64         
 1   district_id   4500 non-null   int64         
 2   frequency     4500 non-null   category      
 3   date_created  4500 non-null   datetime64[ns]
dtypes: category(1), datetime64[ns](1), int64(2)
memory usage: 110.1 KB
None

Erste Zeilen:
   account_id  district_id frequency date_created
0         576           55   monthly   1993-01-01
1        3818           74   monthly   1993-01-01
2         704           55   monthly   1993-01-01
3        2378           16   monthly   1993-01-01
4        2632           24   monthly   1993-01-02

Statistische Übersicht (describe):
          account_id  district_id frequency                date_created
count    4500.000000  4500.0000

📌 Zusammenfassung der Änderungen im DataFrame df_account

1️⃣ Datum in datetime-Format konvertiert  
	•	Die Spalte date wurde in das datetime-Format (%y%m%d) umgewandelt und anschließend in date_created umbenannt.  
	•	Ermöglicht eine einfachere Zeitreihenanalyse.

2️⃣ Spalte district_id in int umgewandelt  
	•	Sicherstellung, dass die district_id als numerischer Wert behandelt wird.

3️⃣ Spalte frequency umbenannt und Werte übersetzt  
	•	POPLATEK MESICNE → monthly  
	•	POPLATEK TYDNE → weekly  
	•	POPLATEK PO OBRATU → after_transaction  

4️⃣ Spalte frequency in eine kategorische Variable (category) umgewandelt  
	•	Spart Speicherplatz und verbessert die Performance bei Analysen.

5️⃣ Spalte date in date_created umbenannt  
	•	Erhöht die Lesbarkeit der Daten.

6️⃣ Überprüfung der account_id-Eindeutigkeit  
	•	account_id wurde auf Einzigartigkeit geprüft, um sicherzustellen, dass es sich um Primärschlüssel handelt.

7️⃣ Analyse des Datenzeitraums  
	•	Es wurde das früheste und späteste date_created geprüft, um den verfügbaren Datenbereich zu verstehen.

### Clients

In [32]:
basic_eda(df_client, df_name="Clients", show_plots=False)


=== [ Clients ] ===
Shape (Rows x Columns): (5369, 3)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5369 entries, 0 to 5368
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   client_id     5369 non-null   int64
 1   birth_number  5369 non-null   int64
 2   district_id   5369 non-null   int64
dtypes: int64(3)
memory usage: 126.0 KB
None

Erste Zeilen:
   client_id  birth_number  district_id
0          1        706213           18
1          2        450204            1
2          3        406009            1
3          4        561201            5
4          5        605703            5

Statistische Übersicht (describe):
          client_id   birth_number  district_id
count   5369.000000    5369.000000  5369.000000
mean    3359.011920  535114.970013    37.310114
std     2832.911984  172895.618429    25.043690
min        1.000000  110820.000000     1.000000
25%     1418.000000  406009.000000    14.000000
50%

In [33]:
# 1) Strings extrahieren
bn_str = df_client["birth_number"].astype(str)

# 2) Jahr, Monat, Tag auslesen
years = bn_str.str[0:2].astype(int)
months = bn_str.str[2:4].astype(int)
days = bn_str.str[4:6].astype(int)

# 3) Geschlecht bestimmen, Monate ggf. korrigieren
is_female = (months > 50)
months[is_female] = months[is_female] - 50  # 50 abziehen für Frauen

# 4) Jahreszahlen von YY in YYYY umwandeln
year_corrected = []
for y in years:
    # Century-Logik (hier: <=19 => 2000+ , sonst 1900+)
    if y <= 19:
        year_corrected.append(y + 2000)
    else:
        year_corrected.append(y + 1900)

# 5) birth_date zusammenbauen
birth_date_list = []
for y, m, d in zip(year_corrected, months, days):
    try:
        birth_date_list.append(pd.Timestamp(y, m, d))
    except ValueError:
        # ungültiges Datum => NaT
        birth_date_list.append(pd.NaT)

df_client["birth_date"] = birth_date_list
df_client["gender"] = pd.Categorical(["F" if f else "M" for f in is_female])

# 6) Alter am 1.1.1999 berechnen
reference_date = pd.Timestamp("1999-01-01")
df_client["alter_1999"] = (reference_date - df_client["birth_date"]).dt.days // 365
# Datentyp ganzzahlig machen:
df_client["alter_1999"] = df_client["alter_1999"].astype("Int64")

# 7) Spalte birth_number droppen
df_client.drop(columns=["birth_number"], inplace=True)

# 8) Geburtsjahre kontrollieren
min_year = df_client["birth_date"].dt.year.min()
max_year = df_client["birth_date"].dt.year.max()
print(f"Geburtsjahre liegen zwischen {min_year} und {max_year}.")

Geburtsjahre liegen zwischen 1920 und 2019.


In [34]:
basic_eda(df_client, df_name="Clients", show_plots=False)


=== [ Clients ] ===
Shape (Rows x Columns): (5369, 5)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5369 entries, 0 to 5368
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   client_id    5369 non-null   int64         
 1   district_id  5369 non-null   int64         
 2   birth_date   5369 non-null   datetime64[ns]
 3   gender       5369 non-null   category      
 4   alter_1999   5369 non-null   Int64         
dtypes: Int64(1), category(1), datetime64[ns](1), int64(2)
memory usage: 178.5 KB
None

Erste Zeilen:
   client_id  district_id birth_date gender  alter_1999
0          1           18 1970-12-13      F          28
1          2            1 1945-02-04      M          53
2          3            1 1940-10-09      F          58
3          4            5 1956-12-01      M          42
4          5            5 1960-07-03      F          38

Statistische Übersicht (describe):
           clie

📌 Zusammenfassung der Änderungen im DataFrame df_client

1️⃣ Extrahieren der birth_number-Daten  
	•	Die Spalte birth_number wurde in einen String (str) umgewandelt, um daraus Jahr, Monat und Tag zu extrahieren.

2️⃣ Jahr, Monat und Tag auslesen  
	•	Die ersten zwei Zeichen (YY) wurden als Jahr (years), die nächsten zwei als Monat (months) und die letzten zwei als Tag (days) interpretiert.

3️⃣ Geschlecht bestimmen & Monatswerte korrigieren  
	•	Frauen haben einen um 50 erhöhten Monatswert (MM+50), also wurde dieser um 50 reduziert.  
	•	Daraus wurde eine neue gender-Spalte ("M" für Männer, "F" für Frauen`) als kategoriale Variable erstellt.

4️⃣ Jahreszahlen von YY in YYYY umgewandelt  
	•	Werte ≤ 19 wurden als 2000+YY, alle anderen als 1900+YY interpretiert.

5️⃣ Geburtsdatum (birth_date) erstellen  
	•	Die bereinigten Jahr-, Monat- und Tag-Werte wurden kombiniert, um echte datetime-Werte zu erstellen.  
	•	Falls das Datum ungültig war (z. B. 31.02.), wurde es als NaT (Not a Time) gesetzt.

6️⃣ Alter zum 01.01.1999 berechnet (alter_1999)  
	•	Differenz zwischen birth_date und Referenzdatum "1999-01-01".  
	•	Umwandlung in eine ganzzahlige (Int64) Spalte, um NaN-Werte zuzulassen.

7️⃣ Spalte birth_number entfernt   
	•	Nachdem alle relevanten Informationen extrahiert wurden, wurde die Originalspalte birth_number gelöscht.

8️⃣ Datenqualität überprüft  
	•	Kontrolle, ob die Geburtsjahre in einem realistischen Bereich liegen (min_year und max_year).

### Dispositionen

In [35]:
basic_eda(df_disp, df_name="Dispositionen", show_plots=False)


=== [ Dispositionen ] ===
Shape (Rows x Columns): (5369, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5369 entries, 0 to 5368
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   disp_id     5369 non-null   int64 
 1   client_id   5369 non-null   int64 
 2   account_id  5369 non-null   int64 
 3   type        5369 non-null   object
dtypes: int64(3), object(1)
memory usage: 167.9+ KB
None

Erste Zeilen:
   disp_id  client_id  account_id       type
0        1          1           1      OWNER
1        2          2           2      OWNER
2        3          3           2  DISPONENT
3        4          4           3      OWNER
4        5          5           3  DISPONENT

Statistische Übersicht (describe):
             disp_id     client_id    account_id   type
count    5369.000000   5369.000000   5369.000000   5369
unique           NaN           NaN           NaN      2
top              NaN           NaN      

In [36]:
# 1) Spalte 'type' zu 'role' umbenennen
df_disp.rename(columns={"type": "role"}, inplace=True)

# 2) 'role' als kategorisch festlegen
df_disp["role"] = df_disp["role"].astype("category")

# 3) Konsistenzprüfung: Gibt es genau 1 Owner pro Konto?

# a) Anzahl Besitzer pro Konto ermitteln
owner_counts = (
    df_disp.loc[df_disp["role"] == "OWNER"]
    .groupby("account_id")["client_id"]
    .count()
)

# b) Wie viele Konten haben KEINEN Owner?
#    (d.h. 'owner_counts' enthält nur Konten mit mindestens 1 Owner,
#     also müssen wir Konten mit 0 Owner anderweitig prüfen)
all_accounts = df_disp["account_id"].unique()
accounts_with_owner = owner_counts.index.unique()
accounts_without_owner = set(all_accounts) - set(accounts_with_owner)
print(f"Anzahl Konten ohne Owner: {len(accounts_without_owner)}")

# c) Wie viele Konten haben MEHR als 1 Owner?
multiple_owner = owner_counts[owner_counts > 1]
print(f"Anzahl Konten mit mehr als 1 Owner: {len(multiple_owner)}")

# d) Bei Bedarf genauer ansehen
if len(accounts_without_owner) > 0:
    print("Beispiele für Konten ohne Owner:", list(accounts_without_owner)[:10])
if len(multiple_owner) > 0:
    print("Beispiele für Konten mit mehreren Ownern:")
    print(multiple_owner.head(10))
    
# Kurzer Überblick über role-Verteilung
print("\nVerteilung der Rollen:")
print(df_disp["role"].value_counts(dropna=False))

Anzahl Konten ohne Owner: 0
Anzahl Konten mit mehr als 1 Owner: 0

Verteilung der Rollen:
role
OWNER        4500
DISPONENT     869
Name: count, dtype: int64


In [37]:
basic_eda(df_disp, df_name="Dispositionen", show_plots=False)


=== [ Dispositionen ] ===
Shape (Rows x Columns): (5369, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5369 entries, 0 to 5368
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   disp_id     5369 non-null   int64   
 1   client_id   5369 non-null   int64   
 2   account_id  5369 non-null   int64   
 3   role        5369 non-null   category
dtypes: category(1), int64(3)
memory usage: 131.3 KB
None

Erste Zeilen:
   disp_id  client_id  account_id       role
0        1          1           1      OWNER
1        2          2           2      OWNER
2        3          3           2  DISPONENT
3        4          4           3      OWNER
4        5          5           3  DISPONENT

Statistische Übersicht (describe):
             disp_id     client_id    account_id   role
count    5369.000000   5369.000000   5369.000000   5369
unique           NaN           NaN           NaN      2
top              NaN       

📌 Zusammenfassung der Änderungen im DataFrame df_disp

1️⃣ Spalte type in role umbenannt  
	•	Erhöht die Verständlichkeit, da role die Funktion eines Kunden im Konto beschreibt.

2️⃣ Spalte role als kategoriale Variable (category) umgewandelt  
	•	Spart Speicherplatz und verbessert die Performance bei Analysen.

3️⃣ Konsistenzprüfung: Gibt es genau 1 Besitzer (OWNER) pro Konto?  
	•	Anzahl der OWNER pro Konto ermittelt.  
	•	Konten ohne OWNER identifiziert und gezählt.  
	•	Konten mit mehreren OWNER aufgedeckt.

4️⃣ Ergebnisse der Prüfung ausgegeben:  
	•	Anzahl der Konten ohne Besitzer.  
	•	Anzahl der Konten mit mehr als einem Besitzer.  
	•	Beispiele für problematische Konten zur manuellen Überprüfung angezeigt.

5️⃣ Übersicht über die Verteilung der Rollen (role) im DataFrame ausgegeben  
	•	Gibt an, wie oft jede Rolle (OWNER, DISPONENT) im Datensatz vorkommt.


### Orders

In [38]:
basic_eda(df_order, df_name="Orders", show_plots=False)


=== [ Orders ] ===
Shape (Rows x Columns): (6471, 6)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6471 entries, 0 to 6470
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   order_id    6471 non-null   int64  
 1   account_id  6471 non-null   int64  
 2   bank_to     6471 non-null   object 
 3   account_to  6471 non-null   int64  
 4   amount      6471 non-null   float64
 5   k_symbol    6471 non-null   object 
dtypes: float64(1), int64(3), object(2)
memory usage: 303.5+ KB
None

Erste Zeilen:
   order_id  account_id bank_to  account_to  amount k_symbol
0     29401           1      YZ    87144583  2452.0     SIPO
1     29402           2      ST    89597016  3372.7     UVER
2     29403           2      QR    13943797  7266.0     SIPO
3     29404           3      WX    83084338  1135.0     SIPO
4     29405           3      CD    24485939   327.0         

Statistische Übersicht (describe):
            order_i

In [39]:
# K_symbol: "POJISTNE" → "insurance", "SIPO" → "household", ...
k_symbol_map = {
    "POJISTNE": "insurance",
    "SIPO": "household",
    "LEASING": "leasing",
    "UVER": "loan_payment",
    " ": "other"  # für leere Einträge
}
df_order["k_symbol"] = df_order["k_symbol"].map(k_symbol_map).astype("category")

# Umbenennungen 
df_order.rename(columns={
    "bank_to": "bank_code_recipient",
    "account_to": "account_number_recipient"
}, inplace=True)

# Umwandlung in kategoriellen Datentyp
df_order["bank_code_recipient"] = df_order["bank_code_recipient"].astype("category")
df_order["k_symbol"] = df_order["k_symbol"].astype("category")

# Prüfen auf Duplikate in order_id (sollte 0 sein)
duplicates = df_order["order_id"].duplicated().sum()
print("Anzahl Duplikate in order_id:", duplicates)


Anzahl Duplikate in order_id: 0


In [40]:
basic_eda(df_order, df_name="Orders", show_plots=False)


=== [ Orders ] ===
Shape (Rows x Columns): (6471, 6)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6471 entries, 0 to 6470
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   order_id                  6471 non-null   int64   
 1   account_id                6471 non-null   int64   
 2   bank_code_recipient       6471 non-null   category
 3   account_number_recipient  6471 non-null   int64   
 4   amount                    6471 non-null   float64 
 5   k_symbol                  6471 non-null   category
dtypes: category(2), float64(1), int64(3)
memory usage: 215.8 KB
None

Erste Zeilen:
   order_id  account_id bank_code_recipient  account_number_recipient  amount  \
0     29401           1                  YZ                  87144583  2452.0   
1     29402           2                  ST                  89597016  3372.7   
2     29403           2                  QR              

📌 Zusammenfassung der Änderungen im DataFrame df_order

1️⃣ Spalte K_symbol übersetzt und kategorisiert  
	•	Werte wurden von Tschechisch ins Englische übersetzt:  
	•	"POJISTNE" → "insurance"
	•	"SIPO" → "household"
	•	"LEASING" → "leasing"
	•	"UVER" → "loan_payment"
	•	Leere Werte (" ") wurden als "other" ersetzt.  
	•	Die Spalte wurde anschließend als kategorische Variable (category) gespeichert.

2️⃣ Spalten umbenannt für bessere Lesbarkeit  
	•	bank_to → bank_code_recipient  
	•	account_to → account_number_recipient  

3️⃣ Weitere Spalten in kategorische Variablen umgewandelt  
	•	bank_code_recipient  
	•	k_symbol  
	•	Spart Speicherplatz und verbessert Analysen.

4️⃣ Prüfung auf doppelte order_id-Werte  
	•	Falls order_id doppelt vorkommt, könnte das auf Datenprobleme hinweisen.  
	•	Ergebnis wird ausgegeben (Anzahl Duplikate).

### Transaktionen

In [41]:
basic_eda(df_trans, df_name="Transaktionen", show_plots=False)


=== [ Transaktionen ] ===
Shape (Rows x Columns): (1056320, 10)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056320 entries, 0 to 1056319
Data columns (total 10 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   trans_id    1056320 non-null  int64  
 1   account_id  1056320 non-null  int64  
 2   date        1056320 non-null  int64  
 3   type        1056320 non-null  object 
 4   operation   873206 non-null   object 
 5   amount      1056320 non-null  float64
 6   balance     1056320 non-null  float64
 7   k_symbol    574439 non-null   object 
 8   bank        273508 non-null   object 
 9   account     295389 non-null   float64
dtypes: float64(3), int64(3), object(4)
memory usage: 80.6+ MB
None

Erste Zeilen:
   trans_id  account_id    date    type operation  amount  balance k_symbol  \
0    695247        2378  930101  PRIJEM     VKLAD   700.0    700.0      NaN   
1    171812         576  930101  PRIJEM     VKLAD   900.0 

In [42]:
# 1) Spalten umbenennen
df_trans.rename(
    columns={
        "type": "trans_type",
        "operation": "trans_operation",
        "account": "partner_account"
    }, 
    inplace=True
)

# 2) Spalte 'date' als Datumsformat parsen
df_trans["date"] = pd.to_datetime(df_trans["date"], format="%y%m%d", errors="coerce")

# 3) Fehlende Werte füllen
df_trans["trans_operation"] = df_trans["trans_operation"].fillna("unknown")
df_trans["k_symbol"] = df_trans["k_symbol"].fillna("NA")
df_trans["bank"] = df_trans["bank"].fillna("NA")

# 4) Tschechisch->Englisch Mapping
mapping_type = {"PRIJEM": "credit", "VYDAJ": "debit"}
df_trans["trans_type"] = df_trans["trans_type"].replace(mapping_type)

mapping_op = {
    "VKLAD": "deposit",
    "VYBER": "withdrawal",
    "VYBER KARTOU": "card_withdrawal",
    "PREVOD NA UCET": "transfer_to_bank",
    "PREVOD Z UCTU": "transfer_from_bank",
    "unknown": "unknown"
}
df_trans["trans_operation"] = df_trans["trans_operation"].replace(mapping_op)

mapping_k = {
    "POJISTNE": "insurance",
    "SIPO": "household",
    "UROK": "interest",
    "SANKC. UROK": "penalty_interest",
    "DUCHOD": "pension",
    "UVER": "loan_payment",
    "NA": "NA"  
}
df_trans["k_symbol"] = df_trans["k_symbol"].replace(mapping_k)

# 5) Jetzt kategorische Umwandlung vornehmen
cat_cols = ["trans_type", "trans_operation", "k_symbol", "bank"]
for col in cat_cols:
    df_trans[col] = df_trans[col].astype("category")

# 6) Numerische Umwandlung
df_trans["partner_account"] = df_trans["partner_account"].astype("Int64")

# 7) Transaktionsjahre kontrollieren
min_year = df_trans["date"].dt.year.min()
max_year = df_trans["date"].dt.year.max()
print(f"Transaktionsjahre liegen zwischen {min_year} und {max_year}.")

Transaktionsjahre liegen zwischen 1993 und 1998.


In [43]:
basic_eda(df_trans, df_name="Transaktionen", show_plots=False)


=== [ Transaktionen ] ===
Shape (Rows x Columns): (1056320, 10)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056320 entries, 0 to 1056319
Data columns (total 10 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   trans_id         1056320 non-null  int64         
 1   account_id       1056320 non-null  int64         
 2   date             1056320 non-null  datetime64[ns]
 3   trans_type       1056320 non-null  category      
 4   trans_operation  1056320 non-null  category      
 5   amount           1056320 non-null  float64       
 6   balance          1056320 non-null  float64       
 7   k_symbol         1056320 non-null  category      
 8   bank             1056320 non-null  category      
 9   partner_account  295389 non-null   Int64         
dtypes: Int64(1), category(4), datetime64[ns](1), float64(2), int64(2)
memory usage: 53.4 MB
None

Erste Zeilen:
   trans_id  account_id       date trans_t

📌 Zusammenfassung der Änderungen im DataFrame df_trans

1️⃣ Spalten umbenannt für bessere Lesbarkeit  
	•	type → trans_type  
	•	operation → trans_operation  
	•	account → partner_account  

2️⃣ Spalte date in ein Datumsformat (datetime) umgewandelt  
	•	"%y%m%d"-Format geparst, um korrekte Zeitwerte zu erhalten.  
	•	Fehlerhafte Werte (errors="coerce") werden als NaT gespeichert.

3️⃣ Fehlende Werte (NaN) ersetzt  
	•	trans_operation → "unknown"  
	•	k_symbol → "NA"  
	•	bank → "NA"

4️⃣ Tschechisch → Englisch Mapping für bessere Interpretierbarkeit  
	•	trans_type:  
	•	"PRIJEM" → "credit"
	•	"VYDAJ" → "debit"
	•	trans_operation:
	•	"VKLAD" → "deposit"
	•	"VYBER" → "withdrawal"
	•	"VYBER KARTOU" → "card_withdrawal"
	•	"PREVOD NA UCET" → "transfer_to_bank"
	•	"PREVOD Z UCTU" → "transfer_from_bank"  
	•	k_symbol:  
	•	"POJISTNE" → "insurance"
	•	"SIPO" → "household"
	•	"UROK" → "interest"
	•	"SANKC. UROK" → "penalty_interest"
	•	"DUCHOD" → "pension"
	•	"UVER" → "loan_payment"

5️⃣ Kategorische Spalten zur Speicheroptimierung konvertiert  
	•	trans_type, trans_operation, k_symbol, bank → category

6️⃣ Spalte partner_account numerisch (Int64) umgewandelt  

7️⃣ Zeitraum der Transaktionsdaten überprüft  
	•	Min. und max. Transaktionsjahr wurden ausgegeben, um sicherzustellen, dass die Daten plausibel sind.

### Loans

In [44]:
basic_eda(df_loan, df_name="Loans", show_plots=False)


=== [ Loans ] ===
Shape (Rows x Columns): (682, 7)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   loan_id     682 non-null    int64  
 1   account_id  682 non-null    int64  
 2   date        682 non-null    int64  
 3   amount      682 non-null    int64  
 4   duration    682 non-null    int64  
 5   payments    682 non-null    float64
 6   status      682 non-null    object 
dtypes: float64(1), int64(5), object(1)
memory usage: 37.4+ KB
None

Erste Zeilen:
   loan_id  account_id    date  amount  duration  payments status
0     5314        1787  930705   96396        12    8033.0      B
1     5316        1801  930711  165960        36    4610.0      A
2     6863        9188  930728  127080        60    2118.0      A
3     5325        1843  930803  105804        36    2939.0      A
4     7240       11013  930906  274740        60    457

In [45]:
# 1) Datum konvertieren
df_loan["date"] = pd.to_datetime(df_loan["date"], format="%y%m%d")
df_loan.rename(columns={"date": "loan_date"}, inplace=True)

# 2) Sinnvolle Umbenennungen
df_loan.rename(columns={
    "status": "loan_status",
    "amount": "loan_amount",
    "payments": "monthly_payment"
}, inplace=True)

# 3) loan_status als category (inkl. Mapping)
status_map = {
    "A": "finished_no_problems",
    "B": "finished_not_paid",
    "C": "running_ok",
    "D": "running_debt"
}
df_loan["loan_status"] = df_loan["loan_status"].replace(status_map).astype("category")

# 4) Datumsbereich prüfen
min_year = df_loan["loan_date"].dt.year.min()
max_year = df_loan["loan_date"].dt.year.max()
print(f"Loanjahre liegen zwischen {min_year} und {max_year}.")

Loanjahre liegen zwischen 1993 und 1998.


In [46]:
basic_eda(df_loan, df_name="Loans", show_plots=False)


=== [ Loans ] ===
Shape (Rows x Columns): (682, 7)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   loan_id          682 non-null    int64         
 1   account_id       682 non-null    int64         
 2   loan_date        682 non-null    datetime64[ns]
 3   loan_amount      682 non-null    int64         
 4   duration         682 non-null    int64         
 5   monthly_payment  682 non-null    float64       
 6   loan_status      682 non-null    category      
dtypes: category(1), datetime64[ns](1), float64(1), int64(4)
memory usage: 33.0 KB
None

Erste Zeilen:
   loan_id  account_id  loan_date  loan_amount  duration  monthly_payment  \
0     5314        1787 1993-07-05        96396        12           8033.0   
1     5316        1801 1993-07-11       165960        36           4610.0   
2     6863        9188 199

📌 Zusammenfassung der Änderungen im DataFrame df_loan

1️⃣ Spalte date in ein Datumsformat (datetime) umgewandelt  
	•	"%y%m%d"-Format geparst und in loan_date umbenannt.  
	•	Erleichtert Zeitreihenanalysen und Berechnungen mit Datumswerten.

2️⃣ Spaltennamen für bessere Verständlichkeit angepasst  
	•	status → loan_status
	•	amount → loan_amount
	•	payments → monthly_payment

3️⃣ Spalte loan_status als kategorische Variable (category) umgewandelt  
	•	Mapping für verständlichere Labels angewendet:  
	•	"A" → "finished_no_problems"
	•	"B" → "finished_not_paid"
	•	"C" → "running_ok"
	•	"D" → "running_debt"

4️⃣ Zeitraum der Loan-Daten überprüft  
	•	Min. und max. Geburtsjahr der Kunden ausgegeben zur Plausibilitätsprüfung der Daten.


### Credit Cards

In [47]:
basic_eda(df_card, df_name="Credit Cards", show_plots=False)


=== [ Credit Cards ] ===
Shape (Rows x Columns): (892, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   card_id  892 non-null    int64 
 1   disp_id  892 non-null    int64 
 2   type     892 non-null    object
 3   issued   892 non-null    object
dtypes: int64(2), object(2)
memory usage: 28.0+ KB
None

Erste Zeilen:
   card_id  disp_id     type           issued
0     1005     9285  classic  931107 00:00:00
1      104      588  classic  940119 00:00:00
2      747     4915  classic  940205 00:00:00
3       70      439  classic  940208 00:00:00
4      577     3687  classic  940215 00:00:00

Statistische Übersicht (describe):
            card_id       disp_id     type           issued
count    892.000000    892.000000      892              892
unique          NaN           NaN        3              607
top             NaN           NaN  classic  9

In [48]:
# 1) Umbenennen der Spalte "type" → "card_type"
df_card.rename(columns={"type": "card_type"}, inplace=True)

# 2) 'issued' in datetime konvertieren
df_card["issued"] = pd.to_datetime(df_card["issued"], format="%y%m%d %H:%M:%S", errors="coerce")

# 3) 'card_type' als Kategorie
df_card["card_type"] = df_card["card_type"].astype("category")

# 4) Datumsbereich prüfen
min_year = df_card["issued"].dt.year.min()
max_year = df_card["issued"].dt.year.max()
print(f"Issue Jahre liegen zwischen {min_year} und {max_year}.")

# 5) Card types prüfen
print(df_card["card_type"].value_counts(dropna=False))

Issue Jahre liegen zwischen 1993 und 1998.
card_type
classic    659
junior     145
gold        88
Name: count, dtype: int64


In [49]:
basic_eda(df_card, df_name="Credit Cards", show_plots=False)


=== [ Credit Cards ] ===
Shape (Rows x Columns): (892, 4)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 892 entries, 0 to 891
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   card_id    892 non-null    int64         
 1   disp_id    892 non-null    int64         
 2   card_type  892 non-null    category      
 3   issued     892 non-null    datetime64[ns]
dtypes: category(1), datetime64[ns](1), int64(2)
memory usage: 22.0 KB
None

Erste Zeilen:
   card_id  disp_id card_type     issued
0     1005     9285   classic 1993-11-07
1      104      588   classic 1994-01-19
2      747     4915   classic 1994-02-05
3       70      439   classic 1994-02-08
4      577     3687   classic 1994-02-15

Statistische Übersicht (describe):
            card_id       disp_id card_type                         issued
count    892.000000    892.000000       892                            892
unique          NaN      

📌 Zusammenfassung der Änderungen im DataFrame df_card

1️⃣ Spalte type in card_type umbenannt  
	•	Erhöht die Verständlichkeit, da die Spalte den Kartentyp beschreibt.

2️⃣ Spalte issued in datetime-Format umgewandelt  
	•	"%y%m%d %H:%M:%S"-Format geparst, um die Ausgabedaten der Karten als Zeitstempel zu speichern.  
	•	Fehlerhafte Werte werden mit errors="coerce" als NaT (Not a Time) gespeichert.

3️⃣ Spalte card_type in eine kategorische Variable (category) umgewandelt  
	•	Spart Speicherplatz und beschleunigt Analysen.

4️⃣ Datumsbereich der ausgegebenen Karten überprüft  
	•	Min. und max. Jahr der Kartenausgabe ausgegeben, um die Plausibilität der Daten sicherzustellen.

5️⃣ Häufigkeit der verschiedenen card_type-Werte geprüft  
	•	Ausgabe der Anzahl jeder Kartentyp-Kategorie zur Analyse der Verteilung.

### Distrikte

In [50]:
basic_eda(df_district, df_name="Distrikte", show_plots=False)


=== [ Distrikte ] ===
Shape (Rows x Columns): (77, 16)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   district_id               77 non-null     int64  
 1   district_name             77 non-null     object 
 2   region                    77 non-null     object 
 3   no_inhabitants            77 non-null     int64  
 4   municipalities_<500       77 non-null     int64  
 5   municipalities_500_1999   77 non-null     int64  
 6   municipalities_2000_9999  77 non-null     int64  
 7   municipalities_>10000     77 non-null     int64  
 8   no_cities                 77 non-null     int64  
 9   urban_ratio               77 non-null     float64
 10  avg_salary                77 non-null     int64  
 11  unemployment_95           77 non-null     object 
 12  unemployment_96           77 non-null     float64
 13  entr

In [51]:
# 1) Umbenennungen
df_district.rename(columns={
    "municipalities_<500": "municipalities_under_500",
    "municipalities_500_1999": "municipalities_500_1999",
    "municipalities_2000_9999": "municipalities_2000_9999",
    "municipalities_>10000": "municipalities_over_10000",
    "unemployment_95": "unemployment_1995",
    "unemployment_96": "unemployment_1996",
    "crimes_95": "crimes_1995",
    "crimes_96": "crimes_1996"
}, inplace=True)

# 2) 'region' als Kategorie
df_district["region"] = df_district["region"].astype("category")

# 3) unemployment_1995 und crimes_1995 als numeric parsen:
df_district["unemployment_1995"] = pd.to_numeric(df_district["unemployment_1995"], errors="coerce")
df_district["crimes_1995"] = pd.to_numeric(df_district["crimes_1995"], errors="coerce")

# 4) Konvertierung in int
df_district["crimes_1995"] = df_district["crimes_1995"].astype("Int64")
df_district["crimes_1996"] = df_district["crimes_1996"].astype("Int64")

# 5) Regionen prüfen
print("Regionen:")
print(df_district["region"].value_counts(dropna=False))


Regionen:
region
south Moravia      14
central Bohemia    12
east Bohemia       11
north Moravia      11
north Bohemia      10
west Bohemia       10
south Bohemia       8
Prague              1
Name: count, dtype: int64


In [52]:
basic_eda(df_district, df_name="Distrikte", show_plots=False)


=== [ Distrikte ] ===
Shape (Rows x Columns): (77, 16)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   district_id                77 non-null     int64   
 1   district_name              77 non-null     object  
 2   region                     77 non-null     category
 3   no_inhabitants             77 non-null     int64   
 4   municipalities_under_500   77 non-null     int64   
 5   municipalities_500_1999    77 non-null     int64   
 6   municipalities_2000_9999   77 non-null     int64   
 7   municipalities_over_10000  77 non-null     int64   
 8   no_cities                  77 non-null     int64   
 9   urban_ratio                77 non-null     float64 
 10  avg_salary                 77 non-null     int64   
 11  unemployment_1995          76 non-null     float64 
 12  unemployment_1996          77 n

📌 Zusammenfassung der Änderungen im DataFrame df_district

1️⃣ Spaltennamen für bessere Verständlichkeit umbenannt  
	•	municipalities_<500 → municipalities_under_500  
	•	municipalities_>10000 → municipalities_over_10000  
	•	unemployment_95 → unemployment_1995  
	•	unemployment_96 → unemployment_1996  
	•	crimes_95 → crimes_1995  
	•	crimes_96 → crimes_1996

2️⃣ Spalte region als kategorische Variable (category) gespeichert  
	•	Spart Speicherplatz und erleichtert die Analyse von Regionen.

3️⃣ Spalten unemployment_1995 und crimes_1995 als numerische Werte geparst  
	•	Verhindert Probleme mit inkonsistenten Datentypen.

4️⃣ Spalten crimes_1995 und crimes_1996 in Integer (Int64) umgewandelt  
	•	Erlaubt NaN-Werte, falls fehlende Daten vorhanden sind.

5️⃣ Regionen geprüft und ihre Verteilung ausgegeben  
	•	Hilft bei der Analyse, wie viele Datenpunkte pro Region vorhanden sind.